# IoU Cannot See the Edges

**Topic:** Boundary metrics &nbsp;·&nbsp; **Difficulty:** Hard

On a large object, getting the boundary badly wrong costs almost nothing in IoU:
the interior is thousands of pixels and the boundary is a few hundred. Two models
can score identically while one traces the outline cleanly and the other returns
a blurry blob.

Boundary IoU fixes this by scoring only a thin band around the contour.
Implement erosion by hand and compare.

## The data (created in the next cell)

- `gt` `(128, 128)`: one large blob.
- `pred_tight`: accurate boundary.
- `pred_blob`: same rough area, sloppy boundary.
- `band = 3` pixels.

## Your task

With **NumPy only** (no `scipy`):

1. Implement 4-connected binary erosion: a pixel survives only if it and its four
   neighbours are all set. Treat outside-the-image as unset.
2. The boundary region of a mask is `mask AND NOT erode^band(mask)`.
3. Compute ordinary IoU and boundary IoU (the IoU of the two boundary regions)
   for both predictions.
4. Set `answers` to a dict with `"iou_tight"`, `"iou_blob"`,
   `"biou_tight"`, `"biou_blob"`, and `"iou_gap"`, `"biou_gap"`, the differences
   `tight - blob` for each metric.

## Evaluation

Compared to the reference to a relative tolerance of 1e-6. The evaluation cell
checks the boundary metric separates the two predictions far more sharply than
plain IoU.

In [ ]:
import numpy as np

rng = np.random.default_rng(3017)
H = W = 128
yy, xx = np.mgrid[0:H, 0:W]
cy, cx = 64.0, 64.0

theta = np.arctan2(yy - cy, xx - cx)
r = np.sqrt((yy - cy) ** 2 + (xx - cx) ** 2)
radius = 36 + 6 * np.sin(5 * theta)              # a wavy outline
gt = r < radius

pred_tight = r < (radius + rng.normal(0, 0.25, (H, W)))
pred_blob = r < 36.6                              # right area, no waviness
band = 3
print(f"gt area {gt.sum()}, tight {pred_tight.sum()}, blob {pred_blob.sum()}")

In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the training data created above to build your model, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.

In [ ]:
# =====================================================================
# EVALUATION: run this cell to score your answer
# =====================================================================
import numpy as np

if "answers" not in dir():
    raise NameError("Define `answers` in the solution cell before running this cell.")
if not isinstance(answers, dict):
    raise TypeError("`answers` must be a dict.")

def _digest(a):
    a = np.asarray(a, dtype=float).ravel()
    if a.size == 0:
        return 0.0
    probe = np.random.default_rng(9176).standard_normal(a.size)
    return float(a @ probe)

_CHECKS = [
    ("iou_tight", [], 0.9877315371662256, -0.16538380775278652, 1e-06),
    ("iou_blob", [], 0.8137254901960784, -0.13624858068238657, 1e-06),
    ("biou_tight", [], 0.8594377510040161, -0.1439025508847503, 1e-06),
    ("biou_blob", [], 0.09974424552429667, -0.016700978459758226, 1e-06),
    ("iou_gap", [], 0.1740060469701472, -0.029135227070399946, 1e-06),
    ("biou_gap", [], 0.7596935054797194, -0.12720157242499205, 1e-06),
]

print("=" * 56)
_ok = True
for _key, _shape, _sum, _probe, _tol in _CHECKS:
    if _key not in answers:
        print(f"  {_key:<24} MISSING")
        _ok = False
        continue
    _a = np.asarray(answers[_key], dtype=float)
    if list(_a.shape) != _shape:
        print(f"  {_key:<24} shape {list(_a.shape)}, expected {_shape}")
        _ok = False
        continue
    if not np.all(np.isfinite(_a)):
        print(f"  {_key:<24} contains NaN or inf")
        _ok = False
        continue
    _g = (abs(_a.sum() - _sum) <= _tol * (1.0 + abs(_sum))
          and abs(_digest(_a) - _probe) <= _tol * (1.0 + abs(_probe)))
    print(f"  {_key:<24} {'ok' if _g else 'wrong'}")
    _ok = _ok and _g

if "biou_gap" in answers:
    _sharper = answers["biou_gap"] > 2 * answers["iou_gap"]
    print(f"  {'boundary IoU separates':<24} {'ok' if _sharper else 'wrong'}  "
          f"(gap {answers['iou_gap']:.4f} -> {answers['biou_gap']:.4f})")
    _ok = _ok and _sharper

print("=" * 56)
if _ok:
    print("PASSED: nice work!")
else:
    print("NOT PASSED yet: revisit your solution and try again.")

## Solution

The blob prediction has almost the same area and almost the same interior, so
plain IoU barely notices that it has thrown away every wave in the outline. The
boundary band is only a few hundred pixels, so restricting the metric to it makes
the difference impossible to hide. This is the motivation for Boundary IoU,
introduced by Cheng et al. in "Boundary IoU: Improving Object-Centric Image
Segmentation Evaluation" (arXiv:2103.16562, CVPR 2021), which is
significantly more sensitive than mask IoU to boundary errors on large
objects, and why "IoU 0.92" on a large-object dataset tells you very little
about outline quality.

```python
import numpy as np

def erode(mask):
    p = np.pad(mask, 1, constant_values=False)
    return (p[1:-1, 1:-1] & p[:-2, 1:-1] & p[2:, 1:-1]
            & p[1:-1, :-2] & p[1:-1, 2:])

def boundary(mask, k):
    e = mask.copy()
    for _ in range(k):
        e = erode(e)
    return mask & ~e

def iou(a, b):
    u = (a | b).sum()
    return float((a & b).sum() / u) if u else 1.0

gb = boundary(gt, band)

iou_tight = iou(gt, pred_tight)
iou_blob = iou(gt, pred_blob)
biou_tight = iou(gb, boundary(pred_tight, band))
biou_blob = iou(gb, boundary(pred_blob, band))

answers = {
    "iou_tight": iou_tight,
    "iou_blob": iou_blob,
    "biou_tight": biou_tight,
    "biou_blob": biou_blob,
    "iou_gap": iou_tight - iou_blob,
    "biou_gap": biou_tight - biou_blob,
}
```